In [ ]:
#!pip install accelerate


[notice] A new release of pip is available: 26.2.2 -> 26.3
[notice] To update, run: C:\Users\soldesk\AppData\Local\Programs\Python\Python314\python.exe -m pip install --upgrade pip


In [1]:
import gradio as gr
from PIL import Image
from transformers import pipeline
import torch

vision = pipeline("image-text-to-text", model="HuggingFaceTB/SmolVLM-256M-Instruct", device_map="auto",dtype=torch.float16)


def echo(message, history):
  # print("message",message) 

  # 사용자 텍스트
    text = message.get("text", "").strip()

    # 이미지 파일 목록
    files = message.get("files", [])

    # 이미지가 있는 경우
    if files:
        image_file = files[0]

        # Gradio 파일 경로 추출
        if isinstance(image_file, dict):
            image_path = image_file.get("path") or image_file.get("name")
        else:
            image_path = image_file

        # 이미지 열기
        image = Image.open(image_path).convert("RGB")

        content = [
            {"type": "image", "image": image},
            {
                "type": "text",
                "text": text if text else "이 이미지를 설명해 주세요."
            }
        ]

    # 텍스트만 입력한 경우
    else:
        content = [
            {
                "type": "text",
                "text": text
            }
        ]

    # 모델 입력 메시지
    messages = [
        {
            "role": "user",
            "content": content
        }
    ]

    # 이미지 분석 및 답변 생성
    result = vision(
        text=messages,
        max_new_tokens=80,
        return_full_text=False
    )

    return result[0]["generated_text"]

demo = gr.ChatInterface(fn=echo, multimodal=True, title="Echo Bot")
demo.launch(debug=True)

c:\source\llm3\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 471/471 [00:00<00:00, 1074.28it/s]


* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


Keyboard interruption in main thread... closing server.
